# 06 Trees homework

Imports

In [1]:
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor

import xgboost as xgb


In [2]:
def rmse(y, y_pred):
    error = y - y_pred
    se = error**2
    mse = se.mean()

    return np.sqrt(mse)

Data preparation

In [3]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
)

df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
display(df.dtypes, df.isnull().sum())

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [5]:
df = df.fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [6]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train["fuel_efficiency_mpg"]
del df_val["fuel_efficiency_mpg"]
del df_test["fuel_efficiency_mpg"]

In [7]:
train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

X_train.shape, X_val.shape

((6000, 16), (2000, 16))

## Question 1

In [8]:
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

features = list(dv.get_feature_names_out())
print(export_text(dt, feature_names=features))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



Answer: model_year

## Question 2

In [9]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse_q2 = rmse(y_val, y_pred)
rmse_q2

np.float64(1.8332197358745623)

Answer: 1.837

## Question 3

In [10]:
scores = []

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    score = rmse(y_val, y_pred)
    scores.append((n, score))

df_scores = pd.DataFrame(scores, columns=["n_estimators", "rmse"])
df_scores.round(3).sort_values("rmse")

,n_estimators,rmse
3,150,1.769
2,100,1.770
1,50,1.774
0,10,1.833


Answer: 150

## Question 4

In [11]:
scores = []

for depth in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(
            n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1
        )
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)
        score = rmse(y_val, y_pred)
        scores.append((depth, n, score))

df_depth_scores = pd.DataFrame(
    scores, columns=["max_depth", "n_estimators", "rmse"]
)
df_depth_scores.pivot(index="n_estimators", columns="max_depth", values="rmse").round(4)

max_depth,10,15,20,25
n_estimators,,,,
10,1.7823,1.8314,1.8373,1.8369
50,1.7475,1.7709,1.7724,1.7721
100,1.7440,1.7679,1.7689,1.7683
150,1.7437,1.7668,1.7685,1.7679


In [12]:
df_depth_scores.groupby("max_depth").rmse.mean().sort_values()

max_depth
10    1.754392
15    1.784224
25    1.786305
20    1.786771
Name: rmse, dtype: float64

Answer: 10

## Question 5

In [13]:
rf = RandomForestRegressor(
    n_estimators=10, max_depth=20, random_state=1, n_jobs=-1
)
rf.fit(X_train, y_train)

df_importance = pd.DataFrame({
    "feature": dv.get_feature_names_out(),
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

df_importance

,feature,importance
9,model_year,0.296107
15,vehicle_weight,0.203313
6,fuel_type=Gasoline,0.160712
8,horsepower,0.078271
4,engine_displacement,0.061281
0,acceleration,0.054998
14,origin=USA,0.042152
2,drivetrain=Front-wheel drive,0.020062
11,num_doors,0.019401
5,fuel_type=Diesel,0.016571


Answer: vehicle_weight

## Question 6

In [14]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, "train"), (dval, "val")]

xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}

In [15]:
model = xgb.train(
    xgb_params, dtrain, num_boost_round=100,
    evals=watchlist, verbose_eval=20
)

y_pred = model.predict(dval)
rmse(y_val, y_pred)

[0]	train-rmse:2.40653	val-rmse:2.46782
[20]	train-rmse:1.39368	val-rmse:1.73766
[40]	train-rmse:1.25326	val-rmse:1.76296
[60]	train-rmse:1.10265	val-rmse:1.79401
[80]	train-rmse:0.99487	val-rmse:1.81308
[99]	train-rmse:0.90106	val-rmse:1.82646


np.float64(1.8264579799904583)

In [16]:
xgb_params["eta"] = 0.1

model = xgb.train(
    xgb_params, dtrain, num_boost_round=100,
    evals=watchlist, verbose_eval=20
)

y_pred = model.predict(dval)
rmse(y_val, y_pred)

[0]	train-rmse:2.74739	val-rmse:2.78635


[20]	train-rmse:1.55847	val-rmse:1.75076


[40]	train-rmse:1.43074	val-rmse:1.71289


[60]	train-rmse:1.37884	val-rmse:1.71688


[80]	train-rmse:1.34578	val-rmse:1.72100


[99]	train-rmse:1.31192	val-rmse:1.72481


np.float64(1.724810059827976)

Compare validation RMSE after all 100 rounds for each eta.

Answer: 0.1